In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
import joblib

In [28]:
file1 = "/content/TrafficCongestion_MultiLocation_7000Rows.xlsx"
file2 = "/content/TrafficCongestion_Sample_Dataset.xlsx"

In [29]:
#Read excel files
df1 = pd.read_excel(file1)
df2 = pd.read_excel(file2)

In [30]:
df = pd.concat([df1, df2], ignore_index=True)

print("Dataset Shape:", df.shape)
print(df.head())

Dataset Shape: (7005, 13)
          Timestamp           Location  Latitude  Longitude  Traffic Volume  \
0  2026-05-01 00:00    Connaught Place   28.6315    77.2167             793   
1  2026-05-01 00:10    Connaught Place   28.6315    77.2167             403   
2  2026-05-01 00:20         India Gate   28.6129    77.2295             120   
3  2026-05-01 00:30  Cyber Hub Gurgaon   28.4959    77.0882             539   
4  2026-05-01 00:40     IGI Airport T3   28.5562    77.1000             337   

   Avg Speed (km/h)     Weather  Rain(mm) Accident Event  \
0                 6        Rain        19       No    No   
1                44         Fog         0       No    No   
2                47       Clear         0      Yes   Yes   
3                 8  Heavy Rain        12      Yes    No   
4                48       Clear         0       No    No   

   Public Transport Density Congestion Level Area  
0                        57        Very High  NaN  
1                        33       

In [31]:
#FEATURE ENGINEERING
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

In [32]:
df['Hour'] = df['Timestamp'].dt.hour
df['DayOfWeek'] = df['Timestamp'].dt.dayofweek

In [33]:
df.drop(columns=['Timestamp'], inplace=True)

In [34]:
label_encoder = LabelEncoder()
df['Congestion Level'] = label_encoder.fit_transform(df['Congestion Level'])

In [35]:
print("Classes:", label_encoder.classes_)

Classes: ['High' 'Low' 'Medium' 'Very High']


In [36]:
X = df.drop(columns=['Congestion Level'])
y = df['Congestion Level']

In [37]:
categorical_cols = X.select_dtypes(include=['object']).columns
numeric_cols = X.select_dtypes(exclude=['object']).columns

print("Categorical Columns:", categorical_cols)
print("Numeric Columns:", numeric_cols)

Categorical Columns: Index(['Location', 'Weather', 'Accident', 'Event', 'Area'], dtype='object')
Numeric Columns: Index(['Latitude', 'Longitude', 'Traffic Volume', 'Avg Speed (km/h)',
       'Rain(mm)', 'Public Transport Density', 'Hour', 'DayOfWeek'],
      dtype='object')


In [38]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

In [39]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [40]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

In [41]:
model = XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',
    eval_metric='mlogloss',
    random_state=42
)

In [42]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

In [43]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

Training Data Shape: (5604, 13)
Testing Data Shape: (1401, 13)


In [44]:
print("Training model...")

pipeline.fit(X_train, y_train)

print("Training completed!")

Training model...
Training completed!


In [45]:
predictions = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print("\nModel Accuracy:", accuracy)
print("\nClassification Report:\n")
print(classification_report(y_test, predictions))


Model Accuracy: 0.9992862241256245

Classification Report:

              precision    recall  f1-score   support

           0       0.99      1.00      1.00       140
           1       1.00      1.00      1.00       482
           2       1.00      1.00      1.00       371
           3       1.00      1.00      1.00       408

    accuracy                           1.00      1401
   macro avg       1.00      1.00      1.00      1401
weighted avg       1.00      1.00      1.00      1401



In [46]:
import os

# Create the 'models' directory if it doesn't exist
if not os.path.exists('models'):
    os.makedirs('models')

joblib.dump(pipeline, 'models/traffic_model.pkl')
joblib.dump(label_encoder, 'models/label_encoder.pkl')

print("\nModel saved successfully!")


Model saved successfully!
